In [14]:
from llama_index.core.indices.multi_modal.base import MultiModalVectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import SimpleDirectoryReader, StorageContext
from llama_index.embeddings.clip import ClipEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import PromptTemplate
from llama_index.multi_modal_llms.ollama import OllamaMultiModal
from llama_index.llms.ollama import Ollama
from llama_index.core.response.notebook_utils import display_query_and_multimodal_response
from llama_index.core import get_response_synthesizer
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.response_synthesizers import BaseSynthesizer
from dotenv import load_dotenv
import os 
import gradio as gr

_ = load_dotenv(".env")

class RAGQueryEngine(CustomQueryEngine):
    """RAG Query Engine."""

    retriever: BaseRetriever
    llm:Ollama
    qa_prompt: PromptTemplate

    def custom_query(self, query: str):
        # Busqueda semantica de nodos relevantes
        nodes = self.retriever.retrieve(query)
        #Extraer el contentido de los nodos
        context_str = "\n\n".join([n.node.get_content() for n in nodes])

        #Generar la respuesta
        response = self.llm.complete(
            self.qa_prompt.format(context_str=context_str, query_str=query)
        )
        return response



def load_documents():
    """Load the context images and text into ImageDocument and Documents"""
    # context images
    image_path = "../datasets_chatbot/images"
    image_documents = SimpleDirectoryReader(image_path).load_data()
    

    # context text
    text_path = "../datasets_chatbot/descriptions"
    text_documents = SimpleDirectoryReader(text_path).load_data()

    return image_documents, text_documents

def create_multimodal_index(image_documents, text_documents):

    # cretate multimodal embed model
    embeded_model = ClipEmbedding()

    # Define the sentence splitter
    node_parser = SentenceSplitter.from_defaults()
    image_nodes = node_parser.get_nodes_from_documents(image_documents)
    text_nodes = node_parser.get_nodes_from_documents(text_documents)

    # Create multimodal index
    index = MultiModalVectorStoreIndex(
    nodes = image_nodes + text_nodes,
    image_embed_model=embeded_model,
    embed_model = embeded_model
    )

    return index


def multimodal_rag(index: MultiModalVectorStoreIndex, llm:str ='llava:13b',k:int=1):

    # Define Prompt Template
    qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "Context: {context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Query: {query_str}\n"
    "Answer: "
)
    # prompt_template = (
    # "Images of shoes are provided.\n"
    # "---------------------\n"
    # "{context}\n"
    # "---------------------\n"
    # "If the images provided cannot help in answering the query\n"
    # "then respond that you are unable to answer the query. Otherwise,\n"
    # "using only the context provided, and not prior knowledge,\n"
    # "provide an answer to the query."
    # "Query: {query}\n"
    # "Answer: "
    # )

    # prompt = PromptTemplate(prompt_template)

    # Instantiate the Ollama MultiModal LLM
    mm_model = Ollama(model=llm, request_timeout=60.0)

    # Define rag query engine (Motor de Búsqueda)
    # rag_engine = index.as_query_engine(
    # llm = mm_model,
    # text_qa_template = prompt
    # )




    # degine retriever
    retriever = index.as_retriever(similarity_top_k=k, image_similarity_top_k=k)

    # instantiate the query engine
    rag_engine = RAGQueryEngine(
    retriever=retriever,
    llm=mm_model,
    qa_prompt=qa_prompt,
    )
    # Retrieve settings. Finde the most relevant document
    rag_engine.retriever.image_similarity_top_k = 1

    return rag_engine


def ingestion():

    # Load the context images and text
    image_documents, text_documents = load_documents()

    # Create index
    index = create_multimodal_index(image_documents, text_documents)

    return index


def rag_respond(query):
    global index
    global rag_engine
    response = rag_engine.query(query)
    return  gr.Image(response.metadata['image_nodes'][0].metadata['file_path']), response.response

   

* Crear el indice multimodal con clip

In [2]:
index = ingestion()

* Busqueda por similitud

In [25]:
#Busqueda por similitud
retriever =index.as_retriever()
retriever.image_similarity_top_k = 1
retriever.similarity_top_k = 1

result = retriever.retrieve("Sneakers")
result

[NodeWithScore(node=TextNode(id_='749b162a-a782-427d-aa7c-2762397a6673', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\5.txt', 'file_name': '5.txt', 'file_type': 'text/plain', 'file_size': 50, 'creation_date': '2025-01-07', 'last_modified_date': '2025-01-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='ef981868-0340-4044-a9c5-94a31d65ea11', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\5.txt', 'file_name': '5.txt', 'file_type': 'text/plain', '

In [26]:
#Busqueda por similitud
result2 = retriever.retrieve("Red sandals")
result2

[NodeWithScore(node=TextNode(id_='4ed64ef4-53b0-47ca-a3d3-3713240fd746', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\4.txt', 'file_name': '4.txt', 'file_type': 'text/plain', 'file_size': 46, 'creation_date': '2025-01-07', 'last_modified_date': '2025-01-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='f908ac6f-5112-4c41-a3fa-88b16bd2410c', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\4.txt', 'file_name': '4.txt', 'file_type': 'text/plain', '

In [27]:
#Busqueda por similitud
result3 = retriever.retrieve("denim boots")
result3

[NodeWithScore(node=TextNode(id_='56a21384-95b4-4772-ac2d-8f812f991506', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\1.txt', 'file_name': '1.txt', 'file_type': 'text/plain', 'file_size': 54, 'creation_date': '2025-01-07', 'last_modified_date': '2025-01-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='8fcd13d2-d653-4f7c-834f-ea9b0e843ffa', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\1.txt', 'file_name': '1.txt', 'file_type': 'text/plain', '

In [30]:
#Busqueda por similitud
result4 = retriever.retrieve("black floral boots")
result4

[NodeWithScore(node=TextNode(id_='8b8af77c-f567-47f2-b18f-e45c62aa8f16', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\2.txt', 'file_name': '2.txt', 'file_type': 'text/plain', 'file_size': 56, 'creation_date': '2025-01-07', 'last_modified_date': '2025-01-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='f722cb8a-9a63-4d57-b24c-8055e315f821', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\2.txt', 'file_name': '2.txt', 'file_type': 'text/plain', '

In [31]:
#Busqueda por similitud
result5 = retriever.retrieve("black and white boots")
result5

[NodeWithScore(node=TextNode(id_='903e66a1-4af2-4769-9908-599e65998781', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\3.txt', 'file_name': '3.txt', 'file_type': 'text/plain', 'file_size': 51, 'creation_date': '2025-01-07', 'last_modified_date': '2025-01-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='8bc60629-67d3-41fe-b1d5-5631c4f275fa', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\3.txt', 'file_name': '3.txt', 'file_type': 'text/plain', '

* Probando el LMM

In [55]:
import ollama
respuesta =ollama.chat(
    model="llava:7b",
    messages=[
        {"role": "user", "content": "¿Cuál es la capital de España?"}],
    stream=False
)
print(respuesta['message']['content'])

 La capital de España es Madrid. 


In [57]:
respuesta =ollama.chat(
    model="llava:7b",
    messages=[
        {"role": "user", "content": "Describe the image.","images": ['../datasets_chatbot/images/1.jpg']}],
    stream=False
)
print(respuesta['message']['content'])

 The image shows a single blue ankle boot with a distressed denim texture, placed against a neutral background. The boot features a decorative toe cap with intricate lace-like patterns and a metallic buckle that runs up the side. A distinctive design element is a patch on the heel of the boot that appears to be crafted from denim material, complementing the overall denim theme of the footwear. The boot's construction suggests durability with a focus on style. The image is a product photo, likely used for commercial purposes such as an online retail listing or an advertisement. There are no texts present in the image. 


* Probando el LMM

Nota: El framework de llama index tiene un bug con la integracion de ollama. No funciona para ningun tipo de dato con esta version.

In [53]:
from PIL import Image
import matplotlib.pyplot as plt

def show_image(image_path):
    """ show image """
    image = Image.open(image_path).convert("RGB")
    plt.figure(figsize=(16, 5))
    plt.imshow(image)

In [58]:
image_documents = SimpleDirectoryReader(
    input_files=["../datasets_chatbot/images/1.jpg"]
).load_data()

# loadd the multimodal model
mm_model = OllamaMultiModal(model="llava:13b", request_timeout=60.0)

# invoke the multimodal model
response = mm_model.complete(
    prompt="Describe the image.",
    image_documents=image_documents,
)

# print the response
print(response)
# show the image
show_image("../datasets_chatbot/images/1.jpg")

AttributeError: 'GenerateResponse' object has no attribute 'items'

In [52]:
from llama_index.llms.ollama import Ollama

llm = Ollama(model="llava:7b", request_timeout=60.0)

response = llm.complete("What is the capital of France?")
print(response)


ValueError: "ChatResponse" object has no field "usage"

nota: no funciona por https://github.com/run-llama/llama_index/issues/17105

*  Usar el indice como query engine

In [45]:
mm_model = OllamaMultiModal(model='llava:7b', request_timeout=60.0)



prompt_template = (
    "Images of shoes are provided.\n"
    "---------------------\n"
    "{context}\n"
    "---------------------\n"
    "If the images provided cannot help in answering the query\n"
    "then respond that you are unable to answer the query. Otherwise,\n"
    "using only the context provided, and not prior knowledge,\n"
    "provide an answer to the query."
    "Query: {query}\n"
    "Answer: "
    )

prompt = PromptTemplate(prompt_template)
rag_engine = index.as_query_engine(
    llm = mm_model,
    prompt = prompt)

rag_engine.retriever.similarity_top_k = 1
rag_engine.retriever.image_similarity_top_k = 1


In [43]:
query = "Show me boots like this one"
image_path = "../datasets_chatbot/images/1.jpg"
response = rag_engine.image_query(prompt_str=query, image_path=image_path)
response

AttributeError: 'GenerateResponse' object has no attribute 'items'

In [82]:
# https://github.com/run-llama/llama_index/issues/17105
from llama_index.core.chat_engine.types import ChatMode
from llama_index.core.agent import AgentRunner
from llama_index.core.tools.query_engine import QueryEngineTool


llm = OllamaMultiModal(model="llava:7b", request_timeout=60.0)
query_engine = index.as_query_engine(llm=llm)
response = query_engine.query("Who is Paul Graham.")
response
# query_engine_tool = QueryEngineTool.from_defaults(query_engine=query_engine)
# agent = AgentRunner.from_llm(
#                 tools=[query_engine_tool],
#                 llm=llm,
#            )
#chat_engine = index.as_chat_engine(llm=llm, chat_mode=ChatMode.REACT)

# response = chat_engine.chat("Tell me a joke.")
# response

AttributeError: 'GenerateResponse' object has no attribute 'items'